# Bài 4: Cross-Validation và Tuning Hyperparameters

## Mục tiêu bài học

Sau bài học này, bạn sẽ có thể:
- Hiểu được tầm quan trọng của việc đánh giá mô hình một cách khách quan và tại sao một lần chia train-test là không đủ.
- Nắm vững khái niệm và quy trình của K-Fold Cross-Validation.
- Sử dụng hàm `lgb.cv` để thực hiện cross-validation một cách hiệu quả với LightGBM.
- Hiểu các chiến lược tìm kiếm siêu tham số (Hyperparameter Tuning) phổ biến: Grid Search, Random Search và Bayesian Optimization.
- Áp dụng `GridSearchCV` hoặc `RandomizedSearchCV` của Scikit-learn để tự động tìm kiếm bộ tham số tốt nhất cho mô hình LightGBM.

## 1. Tại sao một lần chia Train-Test là không đủ?

Trong các bài học trước, chúng ta đã đánh giá mô hình bằng cách chia dữ liệu một lần thành tập huấn luyện (train set) và tập kiểm tra (test set). Phương pháp này đơn giản và nhanh chóng, nhưng có một nhược điểm lớn: **kết quả đánh giá phụ thuộc rất nhiều vào sự may rủi của lần chia đó**.

- Nếu vô tình tập test chứa toàn các mẫu "dễ đoán", chúng ta sẽ có một điểm số lạc quan giả tạo.
- Ngược lại, nếu tập test chứa nhiều mẫu ngoại lệ hoặc khó, điểm số sẽ thấp một cách không công bằng.

Điều này làm cho việc đánh giá mô hình trở nên không ổn định và không đáng tin cậy. Để giải quyết vấn đề này, chúng ta sử dụng **Cross-Validation (Kiểm định chéo)**.

## 2. K-Fold Cross-Validation: Tiêu chuẩn vàng trong đánh giá mô hình

**K-Fold Cross-Validation** là kỹ thuật phổ biến nhất. Ý tưởng rất đơn giản:

1.  **Chia dữ liệu:** Chia toàn bộ tập dữ liệu huấn luyện thành K "phần" (fold) bằng nhau (ví dụ: K=5 hoặc K=10).
2.  **Lặp K lần:**
    - Ở mỗi lần lặp, chọn **một phần** làm tập validation (để đánh giá).
    - **K-1 phần còn lại** được dùng làm tập train.
    - Huấn luyện mô hình trên tập train và đánh giá trên tập validation.
3.  **Tổng hợp kết quả:** Kết quả cuối cùng là **trung bình** của các điểm số thu được từ K lần lặp. Điểm số này sẽ ổn định và đáng tin cậy hơn nhiều so với một lần chia duy nhất.

![K-Fold Cross-Validation](https://scikit-learn.org/stable/_images/grid_search_cross_validation.png)
*Nguồn: Scikit-learn Documentation*

### Sử dụng `lgb.cv`: Công cụ Cross-Validation tích hợp

LightGBM cung cấp một hàm `lgb.cv` rất tiện lợi để thực hiện cross-validation mà không cần vòng lặp thủ công.

In [ ]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Sử dụng lại dữ liệu Churn từ các bài trước
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)
df.drop(columns=['customerID'], inplace=True)
for column in df.select_dtypes(include=['object']).columns:
    df[column] = LabelEncoder().fit_transform(df[column])

X = df.drop('Churn', axis=1)
y = df['Churn']

# Tạo đối tượng Dataset của LightGBM
lgb_data = lgb.Dataset(X, label=y)

In [ ]:
# Định nghĩa các tham số cho mô hình
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'n_estimators': 2000, # Số cây lớn để early stopping hoạt động
    'learning_rate': 0.01,
    'num_leaves': 20,
    'max_depth': 5,
    'seed': 42,
    'n_jobs': -1,
    'verbose': -1,
    'colsample_bytree': 0.7,
    'subsample': 0.7
}

# Chạy Cross-Validation với 5 folds
cv_results = lgb.cv(
    params=params,
    train_set=lgb_data,
    num_boost_round=params['n_estimators'], # Tương đương n_estimators
    nfold=5, # Số folds
    stratified=True, # Giữ nguyên tỷ lệ lớp trong mỗi fold (quan trọng cho dữ liệu mất cân bằng)
    shuffle=True,
    metrics='auc', # Metric để theo dõi
    callbacks=[lgb.early_stopping(100, verbose=False)], # Dừng sớm nếu không cải thiện
    seed=42
)

print(f"Điểm AUC trung bình tốt nhất trên 5 folds: {cv_results['valid auc-mean'][-1]:.4f}")
print(f"Độ lệch chuẩn của điểm AUC: {cv_results['valid auc-stdv'][-1]:.4f}")
print(f"Số cây tối ưu tìm được: {len(cv_results['valid auc-mean'])}")

`lgb.cv` trả về một dictionary chứa lịch sử điểm số trung bình và độ lệch chuẩn qua các vòng lặp. Kết quả này cho chúng ta một ước tính đáng tin cậy về hiệu suất của mô hình với bộ tham số đã cho.

## 3. Tự động tìm kiếm Siêu tham số (Hyperparameter Tuning)

Chúng ta đã biết cách đánh giá một bộ tham số. Nhưng làm thế nào để tìm ra bộ tham số *tốt nhất* từ hàng ngàn, hàng triệu khả năng kết hợp? Đây là lúc các thuật toán tìm kiếm tự động phát huy tác dụng.

### a) Grid Search (Tìm kiếm lưới)

- **Ý tưởng:** Bạn định nghĩa một "lưới" các giá trị cho mỗi tham số bạn muốn tinh chỉnh. Grid Search sẽ thử **tất cả các kết hợp có thể có** của các giá trị này.
- **Ví dụ:**
  - `num_leaves`: [20, 30, 40]
  - `learning_rate`: [0.1, 0.05]
  - Grid Search sẽ thử 3 * 2 = 6 bộ tham số.
- **Ưu điểm:** Đảm bảo sẽ tìm ra bộ tham số tốt nhất trong lưới bạn định nghĩa.
- **Nhược điểm:** Cực kỳ tốn kém về mặt tính toán. Nếu bạn có nhiều tham số hoặc nhiều giá trị cho mỗi tham số, số lượng kết hợp sẽ bùng nổ (tổ hợp bùng nổ - combinatorial explosion).

### b) Random Search (Tìm kiếm ngẫu nhiên)

- **Ý tưởng:** Thay vì thử tất cả các kết hợp, Random Search sẽ chọn ngẫu nhiên một số lượng `n_iter` các bộ tham số từ không gian bạn định nghĩa.
- **Ví dụ:** Bạn định nghĩa một khoảng giá trị (phân phối) cho mỗi tham số:
  - `num_leaves`: từ 10 đến 50
  - `learning_rate`: từ 0.01 đến 0.2
  - Random Search sẽ chọn ngẫu nhiên 10 (hoặc `n_iter`) bộ tham số trong các khoảng này.
- **Ưu điểm:** Hiệu quả hơn Grid Search rất nhiều. Thường thì chỉ cần một số lượng lần thử vừa phải là đã có thể tìm ra một bộ tham số rất tốt. Nó không lãng phí thời gian vào những tham số không quan trọng.
- **Nhược điểm:** Không đảm bảo tìm ra bộ tham số tốt nhất tuyệt đối.

### c) Bayesian Optimization (Tối ưu hóa Bayes)

- **Ý tưởng:** Đây là một phương pháp thông minh hơn. Nó bắt đầu bằng cách thử một vài bộ tham số ngẫu nhiên. Sau đó, nó sử dụng kết quả của các lần thử trước để xây dựng một "mô hình xác suất" (surrogate model) của hàm mục tiêu. Dựa trên mô hình này, nó sẽ quyết định bộ tham số tiếp theo nào có khả năng mang lại cải thiện lớn nhất.
- **Ưu điểm:** Thông minh và hiệu quả hơn Random Search. Nó không tìm kiếm một cách mù quáng mà học hỏi từ các lần thử trước. Thường tìm ra bộ tham số tốt hơn với số lần thử ít hơn.

: 
,
: {},
: [

: 
,
: null,
: {},
: [],
: [
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
,
Bắt đầu tìm kiếm siêu tham số...")
random_search.fit(X, y)

print("\nTìm kiếm hoàn tất!")
print(f"Bộ tham số tốt nhất tìm được: {random_search.best_params_}")
print(f"Điểm ROC AUC tốt nhất trên cross-validation: {random_search.best_score_:.4f}")

## Tóm tắt

- **Cross-Validation** là kỹ thuật cần thiết để đánh giá mô hình một cách khách quan, đáng tin cậy.
- `lgb.cv` là một công cụ mạnh mẽ để thực hiện cross-validation nhanh chóng với LightGBM.
- **Hyperparameter Tuning** là quá trình tự động tìm kiếm bộ tham số tốt nhất.
- **Grid Search** thử tất cả các kết hợp, chính xác nhưng chậm.
- **Random Search** thử ngẫu nhiên, hiệu quả hơn và thường là lựa chọn tốt để bắt đầu.
- **Bayesian Optimization** là phương pháp thông minh, học hỏi từ các lần thử trước, thường cho kết quả tốt nhất với ít lần thử nhất.
- Bạn có thể kết hợp LightGBM với `RandomizedSearchCV` của Scikit-learn để có một quy trình tuning mạnh mẽ.

## Bài học tiếp theo

Một mô hình có điểm số cao là rất tốt, nhưng liệu chúng ta có thể tin tưởng nó không? Làm thế nào để giải thích cho sếp hoặc khách hàng tại sao mô hình lại đưa ra một dự đoán cụ thể? Trong bài học cuối cùng, chúng ta sẽ khám phá các kỹ thuật để phân tích và diễn giải mô hình LightGBM.